# Module 8A: DQ Dashboard - Snowflake Native Dashboards + Power BI

## Learning Objectives
- Create BI-ready DQ reporting views
- Build a dashboard using **Snowflake Dashboards** (SQL query tiles)
- Understand Power BI best practices for DQ views

> **Business Value:** Native dashboards = zero additional infrastructure. Same platform that runs DQ checks also displays results.

---
> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes | **Variant:** Native + Power BI

> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE DQ_LAB_WH;

---
## Shared Setup: Create DQ Reporting Views

> **Note:** These views are identical across all Module 8 variants (8A/8B/8C). If you already ran another variant, these views already exist -- running them again is safe (`CREATE OR REPLACE`).
>
> **Business Value:** BI tools cannot call table functions directly. Views provide a stable, queryable interface.

In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_RESULTS_FLAT AS
SELECT
    r.REF_ENTITY_NAME AS TABLE_NAME, r.METRIC_NAME,
    r.ARGUMENT_NAMES AS COLUMN_CHECKED, r.VALUE AS METRIC_VALUE,
    r.EXPECTATION_NAME, r.EXPECTATION_RESULT, r.MEASUREMENT_TIME,
    COALESCE(c.SEVERITY, 'MEDIUM') AS SEVERITY,
    COALESCE(c.OWNER, 'Unassigned') AS RULE_OWNER,
    COALESCE(c.RULE_TYPE, 'SYSTEM') AS RULE_TYPE,
    CASE WHEN r.EXPECTATION_RESULT = 'MET' THEN 'PASS'
         WHEN r.EXPECTATION_RESULT = 'NOT_MET' THEN 'FAIL'
         ELSE 'NO_EXPECTATION' END AS STATUS
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'
)) r
LEFT JOIN CORP_DWH.DQ.RULES_CATALOG c
    ON UPPER(r.METRIC_NAME) LIKE '%' || REPLACE(UPPER(c.RULE_NAME), ' ', '_') || '%';

> **What this does:** Creates V_DQ_SCORECARD view that calculates per-table health scores from the latest DQ expectation results.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_SCORECARD AS
WITH latest AS (
    SELECT 'CORP_DWH.GOLD.DIM_CUSTOMER' AS TABLE_NAME,
        METRIC_NAME, ARGUMENT_NAMES, VALUE, EXPECTATION_NAME, EXPECTATION_RESULT, MEASUREMENT_TIME,
        ROW_NUMBER() OVER (PARTITION BY METRIC_NAME, ARGUMENT_NAMES ORDER BY MEASUREMENT_TIME DESC) AS RN
    FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
    WHERE EXPECTATION_NAME IS NOT NULL
)
SELECT TABLE_NAME,
    COUNT(*) AS TOTAL_EXPECTATIONS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSED,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILED,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS HEALTH_SCORE_PCT,
    MAX(MEASUREMENT_TIME) AS LAST_EVALUATED
FROM latest WHERE RN = 1 GROUP BY TABLE_NAME;

> **What this does:** Creates V_DQ_TREND view that provides hourly time-series data for charting quality metrics over time.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_TREND AS
SELECT DATE_TRUNC('HOUR', MEASUREMENT_TIME) AS MEASUREMENT_HOUR,
    METRIC_NAME, VALUE AS METRIC_VALUE, EXPECTATION_RESULT, MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL ORDER BY MEASUREMENT_TIME DESC;

> **What this does:** Creates V_DQ_EXECUTIVE_SUMMARY view that aggregates all checks into a single overall health percentage.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY AS
SELECT 'CORP_DWH' AS DATA_ESTATE, COUNT(*) AS TOTAL_CHECKS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS CHECKS_PASSING,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS CHECKS_FAILING,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS OVERALL_HEALTH_PCT,
    CURRENT_TIMESTAMP() AS AS_OF
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL;

---
## Snowflake Dashboard Tiles

Go to **Projects > Dashboards > + Dashboard** and add these SQL tiles:

### Tile 1: Health KPI (Scorecard)

In [ ]:
SELECT OVERALL_HEALTH_PCT AS "Data Quality Health (%)" FROM CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY;

### Tile 2: Pass vs Fail (Bar Chart)

In [ ]:
SELECT STATUS, COUNT(*) AS CHECK_COUNT
FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT
WHERE STATUS IN ('PASS', 'FAIL')
GROUP BY STATUS;

### Tile 3: Failures by Severity (Horizontal Bar)

In [ ]:
SELECT SEVERITY, COUNT(*) AS FAILURE_COUNT
FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT
WHERE STATUS = 'FAIL'
GROUP BY SEVERITY
ORDER BY CASE SEVERITY WHEN 'CRITICAL' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MEDIUM' THEN 3 ELSE 4 END;

### Tile 4: Top Failing Rules (Table)

In [ ]:
SELECT METRIC_NAME AS RULE, COLUMN_CHECKED, METRIC_VALUE AS VIOLATIONS, SEVERITY, RULE_OWNER
FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT
WHERE STATUS = 'FAIL' AND METRIC_VALUE > 0
ORDER BY METRIC_VALUE DESC LIMIT 10;

### Tile 5: Trend (Line Chart)

In [ ]:
SELECT MEASUREMENT_HOUR,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSING,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILING
FROM CORP_DWH.DQ.V_DQ_TREND
GROUP BY MEASUREMENT_HOUR ORDER BY MEASUREMENT_HOUR;

---
## Power BI Integration: Step-by-Step How-To

### Step 1: Install the Snowflake Connector

1. Open **Power BI Desktop** (download from [powerbi.microsoft.com](https://powerbi.microsoft.com/desktop))
2. Click **Get Data** > search "Snowflake"
3. Select **Snowflake** connector > click **Connect**
4. Enter your connection details:

| Field | Value |
|-------|-------|
| Server | `<your-account>.snowflakecomputing.com` |
| Warehouse | `DQ_LAB_WH` |
| Database | `CORP_DWH` (optional, narrows scope) |

5. Choose **DirectQuery** (recommended for DQ dashboards -- always shows live data)
6. Authenticate with your Snowflake credentials (or SSO if configured)

### Step 2: Select the DQ Views

In the Navigator panel, expand: `CORP_DWH` > `DQ` schema. Select these 4 views:

| View | Load Mode | Why |
|------|-----------|-----|
| `V_DQ_EXECUTIVE_SUMMARY` | DirectQuery | 1 row, always fresh |
| `V_DQ_SCORECARD` | DirectQuery | Small, changes on each DMF run |
| `V_DQ_RESULTS_FLAT` | DirectQuery | Core data, needs latest results |
| `V_DQ_TREND` | Import (schedule hourly) | Historical, grows over time |

> **Why mix modes?** DirectQuery hits Snowflake on every interaction (always fresh, but slower). Import caches locally (fast, but stale until refresh). For a DQ dashboard, freshness matters more than speed.

### Step 3: Build the Dashboard Layout

**Recommended 5-tile layout:**

```
┌─────────────────────────────────────────────────────────────┐
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐   │
│  │ Health % │  │  Total   │  │ Passing  │  │ Failing  │   │
│  │  (Card)  │  │ (Card)   │  │ (Card)   │  │ (Card)   │   │
│  └──────────┘  └──────────┘  └──────────┘  └──────────┘   │
├─────────────────────────────────┬───────────────────────────┤
│                                 │                           │
│   Pass/Fail by Severity         │    Health Trend           │
│   (Stacked Bar Chart)           │    (Line Chart)           │
│                                 │                           │
├─────────────────────────────────┴───────────────────────────┤
│                                                             │
│   Failing Rules Detail (Matrix/Table)                       │
│   Columns: Rule, Column, Violations, Severity, Owner       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Step 4: Configure Each Visual

**KPI Cards (top row):**
- Drag `OVERALL_HEALTH_PCT` from V_DQ_EXECUTIVE_SUMMARY to a Card visual
- Format: Data label > Display units: None, Decimal places: 0, Suffix: "%"
- Repeat for `TOTAL_CHECKS`, `CHECKS_PASSING`, `CHECKS_FAILING`
- Conditional formatting: Red if health < 70%, yellow if < 90%, green if ≥ 90%

**Stacked Bar (left):**
- Source: `V_DQ_RESULTS_FLAT`
- Axis: `SEVERITY` (order: CRITICAL > HIGH > MEDIUM > LOW)
- Values: Count of rows
- Legend: `STATUS` (green = PASS, red = FAIL)
- Filter: Exclude `NO_EXPECTATION` status

**Trend Line (right):**
- Source: `V_DQ_TREND`
- X-axis: `MEASUREMENT_HOUR`
- Values: Count where `EXPECTATION_RESULT = 'MET'` (Passing) and `NOT_MET` (Failing)
- Colors: Green for passing, red for failing

**Detail Table (bottom):**
- Source: `V_DQ_RESULTS_FLAT`
- Columns: `METRIC_NAME`, `COLUMN_CHECKED`, `METRIC_VALUE`, `SEVERITY`, `RULE_OWNER`
- Filter: `STATUS = 'FAIL'` and `METRIC_VALUE > 0`
- Sort: `METRIC_VALUE` descending
- Conditional formatting: Red background on CRITICAL severity rows

### Step 5: Add DAX Measures

In the Modeling tab, create these calculated measures:

```dax
-- Overall health percentage (useful for conditional formatting)
Health Score = 
DIVIDE(
    COUNTROWS(FILTER('V_DQ_RESULTS_FLAT', [STATUS] = "PASS")),
    COUNTROWS('V_DQ_RESULTS_FLAT')
) * 100

-- Count of critical failures (for alert KPI)
Critical Failures = 
COUNTROWS(
    FILTER('V_DQ_RESULTS_FLAT', 
        [STATUS] = "FAIL" && [SEVERITY] = "CRITICAL")
)

-- Failure rate trend (for sparklines)
Failure Rate = 
DIVIDE(
    COUNTROWS(FILTER('V_DQ_TREND', [EXPECTATION_RESULT] = "NOT_MET")),
    COUNTROWS('V_DQ_TREND')
) * 100

-- Time since last evaluation (staleness indicator)
Hours Since Last Check = 
DATEDIFF(
    MAX('V_DQ_EXECUTIVE_SUMMARY'[AS_OF]),
    NOW(),
    HOUR
)
```

### Step 6: Set Up Row-Level Security (Optional)

If different teams should only see their own rules:

1. Modeling tab > **Manage Roles** > New Role: `TeamFinance`
2. Table: `V_DQ_RESULTS_FLAT`, DAX filter: `[RULE_OWNER] = "Finance Team"`
3. Repeat for each team
4. Publish to Power BI Service > Workspace > Dataset > Security > Add members

This maps Power BI roles to the `RULE_OWNER` column from the Rules Catalog.

### Step 7: Schedule Refresh (Power BI Service)

After publishing to Power BI Service:

1. Go to Dataset settings > **Scheduled refresh**
2. Gateway: Use Snowflake gateway or configure **Snowflake cloud connector**
3. Credentials: Enter Snowflake username/password (or OAuth)
4. Schedule: Every 1 hour for Import tables (V_DQ_TREND)
5. DirectQuery tables refresh automatically on each dashboard interaction

### Tips

| Tip | Details |
|-----|---------|
| Use a dedicated role | Create `CORP_DQ_STEWARD` with SELECT on DQ schema only |
| Bookmark alerts | Create bookmarks for "All Passing" vs "Has Failures" states |
| Mobile layout | Power BI mobile app works well with KPI cards at top |
| Embed in Teams | Pin the dashboard to a Teams channel for visibility |
| Alerts | Set data alerts on `Critical Failures` measure > notify on Teams/email |


---
## Checkpoint

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("=" * 50)
print("CHECKPOINT: Dashboard Views Ready")
print("=" * 50)
for v in ['V_DQ_RESULTS_FLAT', 'V_DQ_SCORECARD', 'V_DQ_TREND', 'V_DQ_EXECUTIVE_SUMMARY']:
    try:
        cnt = session.sql(f"SELECT COUNT(*) AS C FROM CORP_DWH.DQ.{v}").collect()[0]['C']
        print(f"  [PASS] {v} -- {cnt} rows")
    except Exception as e:
        print(f"  [FAIL] {v}: {str(e)[:50]}")
print("=" * 50)

---
**Next:** Try 8B (Python charts) or 8C (Streamlit), or proceed to `9_TEARDOWN`.